# Прогноз остатков физлиц в BYN

Подготовьте в ядре pandas DataFrame `df` с колонками `date` и `amount`. `date` содержит календарные даты, `amount` — остатки в BYN.

Ноутбук сравнивает наивные модели, ARIMA, SARIMA и SARIMAX с календарём. Критерий выбора — MAE на горизонтах 1–30 дней с учётом лимита. Последние шесть календарных месяцев выделены под test.

## 1. Настройки

Если пакетов ещё нет в ядре, выполните `%pip install pandas numpy matplotlib "statsmodels>=0.14.6,<0.16"`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import ConvergenceWarning, InterpolationWarning
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

VALID_MONTHS = 3
TEST_MONTHS = 6
MAE_HORIZON = 30
SCALE = 1_000_000
LIMIT_BYN = 30_000_000
MAX_STREAK = 7
ALPHA = .05
WEEKDAYS = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]
FIXED_LEADS = (1, 7, 14, 30, 90, 180)
BUCKET_ENDS = (7, 14, 30, 90, 180, 366)

plt.rcParams.update({"figure.figsize": (11, 3.4), "axes.grid": True, "grid.alpha": .22})

## 2. Календарь, разделение и масштаб

Пропуски дат остаются на дневной шкале как `NaN`. SARIMA обрабатывает их в фильтре Калмана, а в MAE участвуют только известные факты.

In [ ]:
def prepare_daily(frame):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("df должен быть pandas DataFrame.")
    if frame.empty or not frame.columns.is_unique or not {"date", "amount"} <= set(frame.columns):
        raise ValueError("df должен содержать строки и уникальные колонки date и amount.")

    date = frame["date"]
    if not pd.api.types.is_datetime64_any_dtype(date):
        iso_date = date.astype("string").str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
        if not iso_date.fillna(False).all():
            raise ValueError("Строковые даты должны иметь формат YYYY-MM-DD.")
        date = pd.to_datetime(date, format="%Y-%m-%d", errors="coerce")
    if date.isna().any() or date.dt.tz is not None or not date.eq(date.dt.normalize()).all():
        raise ValueError("Нужны существующие календарные даты без времени и часового пояса.")
    if date.duplicated().any():
        duplicates = date[date.duplicated()].dt.strftime("%Y-%m-%d").head().tolist()
        raise ValueError(f"Найдены повторяющиеся даты: {duplicates}")

    amount = pd.to_numeric(frame["amount"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    if not np.isfinite(amount).all():
        raise ValueError("В amount нужны конечные числовые остатки в BYN.")
    return pd.Series(amount, index=pd.DatetimeIndex(date, name="date"), name="amount").sort_index().asfreq("D")


if "df" not in globals():
    raise NameError("Сначала подготовьте df в этом ядре.")
daily = prepare_daily(df)

In [ ]:
as_of = daily.index[-1]
test_cut = as_of - pd.DateOffset(months=TEST_MONTHS)
train_cut = test_cut - pd.DateOffset(months=VALID_MONTHS)

train_byn = daily.loc[:train_cut]
valid_byn = daily.loc[train_cut + pd.Timedelta(days=1):test_cut]
test_byn = daily.loc[test_cut + pd.Timedelta(days=1):]
splits = {"train": train_byn, "valid": valid_byn, "test": test_byn}

if train_byn.count() < 60 or valid_byn.count() < 30 or test_byn.count() < 30:
    raise ValueError("Нужно минимум 60 фактов в train и по 30 фактов в valid и test.")

display(pd.DataFrame({
    name: {"от": part.index[0].date(), "до": part.index[-1].date(),
           "календарных дней": len(part), "фактов": part.count(), "пропусков": part.isna().sum()}
    for name, part in splits.items()
}).T)

Из каждого остатка вычитаем медиану train и делим на 1 млн. Так модель работает с отклонениями от типичного уровня. Это линейное преобразование, поэтому прогноз можно точно вернуть в исходную размерность. MAE и порог считаются в BYN.

In [ ]:
train_center_byn = train_byn.median()
target = (daily - train_center_byn) / SCALE
train = target.loc[train_byn.index]
valid = target.loc[valid_byn.index]
test = target.loc[test_byn.index]

display(pd.Series({
    "центр train, BYN": train_center_byn,
    "медиана масштабированного train": train.median(),
    "первый квартиль train": train.quantile(.25),
    "третий квартиль train": train.quantile(.75),
}).round(2))

## 3. EDA на train

In [ ]:
delta_mln = train.diff()
balance_mln = train_byn / SCALE

display(pd.DataFrame({"остаток, млн BYN": balance_mln, "изменение, млн BYN": delta_mln})
        .describe(percentiles=[.01, .05, .50, .95, .99]).round(2))
largest_days = delta_mln.abs().nlargest(10).index
display(delta_mln.loc[largest_days].rename("крупнейшие изменения, млн BYN").round(2).to_frame())

In [ ]:
fig, ax = plt.subplots()
ax.plot(balance_mln, color="#1d6981", label="Факт")
ax.plot(balance_mln.rolling(30, min_periods=15).median(), color="#c95926", label="Медиана за 30 дней")
ax.set(title="Остаток на train", ylabel="млн BYN")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(delta_mln, color="#1d6981", linewidth=.9)
ax.axhline(0, color="#555555", linewidth=.8)
ax.set(title="Изменение к предыдущему календарному дню", ylabel="млн BYN")
plt.show()

In [ ]:
weekday_coverage = train.notna().groupby(train.index.day_of_week).mean().reindex(range(7))
fig, ax = plt.subplots()
ax.bar(WEEKDAYS, 100 * weekday_coverage, color="#1d6981")
ax.set(title="Доля дат с фактом по дням недели", ylabel="%", ylim=(0, 105))
plt.show()

Для календарных профилей вычитаем локальную 63-дневную медиану. Это диагностическое сглаживание используется только внутри train. Сравнение первой и второй половин помогает увидеть, повторяется ли рисунок.

In [ ]:
detrended = train - train.rolling(63, center=True, min_periods=24).median()
mid = len(detrended) // 2
eda_parts = {"Первая половина": detrended.iloc[:mid], "Вторая половина": detrended.iloc[mid:]}

weekly_profiles = pd.DataFrame({
    name: part.groupby(part.index.day_of_week).median()
    for name, part in eda_parts.items()
}).reindex(range(7))
weekly_profiles.index = WEEKDAYS
weekday_counts = train.groupby(train.index.day_of_week).count().reindex(range(7), fill_value=0)
display(weekly_profiles.assign(наблюдений=weekday_counts.to_numpy()).round(2))

In [ ]:
fig, ax = plt.subplots()
weekly_profiles.plot(ax=ax, marker="o", color=["#1d6981", "#c95926"])
ax.set(title="Недельный профиль после удаления локального уровня", ylabel="млн BYN")
plt.show()

In [ ]:
month_profile = pd.DataFrame({
    name: part.groupby(part.index.day).median()
    for name, part in eda_parts.items()
})
fig, ax = plt.subplots()
month_profile.plot(ax=ax, marker=".", color=["#1d6981", "#c95926"])
ax.set(title="Профиль внутри месяца", xlabel="День месяца", ylabel="млн BYN")
plt.show()

In [ ]:
last_week = detrended.loc[(detrended.index.days_in_month - detrended.index.day) < 7]
days_to_end = last_week.index.days_in_month - last_week.index.day
end_month_profile = last_week.groupby(days_to_end).agg(["median", "count"])
display(end_month_profile.rename_axis("дней до конца месяца")
        .rename(columns={"median": "медиана, млн BYN", "count": "фактов"}).round(2))

## 4. Статистические тесты

Для тестов, которым нужна регулярная последовательность без пропусков, используем самый длинный непрерывный участок train. Если факты есть только по будням, для диагностики используем шкалу понедельник–пятница и сохраняем пропуски внутри неё. При коротком участке выводы тестов пропускаются. Прогноз и проверка лимита по-прежнему идут по календарным дням.

In [ ]:
def longest_complete_block(series):
    blocks = (block.dropna() for _, block in series.groupby(series.isna().cumsum()))
    return max(blocks, key=len, default=series.iloc[:0])


diagnostic = longest_complete_block(train)
diagnostic_calendar = "календарные дни"
weekdays_only = (train.dropna().index.day_of_week < 5).all()
if len(diagnostic) < 40 and weekdays_only:
    diagnostic = longest_complete_block(train.asfreq("B"))
    diagnostic_calendar = "шкала понедельник–пятница"

if len(diagnostic):
    print(f"Диагностика: {diagnostic_calendar}, {len(diagnostic)} наблюдений, "
          f"с {diagnostic.index[0].date()} по {diagnostic.index[-1].date()}.")

**ADF** проверяет нулевую гипотезу о единичном корне. **KPSS** проверяет нулевую гипотезу о стационарности вокруг постоянного уровня. Смотрим оба теста на уровне и первой разности. У KPSS значения 0,01 и 0,10 могут быть границами таблицы. Порядок модели выбираем на valid. [Документация ADF и KPSS](https://www.statsmodels.org/stable/examples/notebooks/generated/stationarity_detrending_adf_kpss.html)

In [ ]:
def stationarity_tests(series):
    if len(series) < 40 or series.nunique() <= 1:
        return {"статус": "мало данных или постоянный ряд"}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", InterpolationWarning)
            warnings.filterwarnings("ignore", r"(adfuller|kpss) currently returns", FutureWarning)
            adf_stat, adf_p, *_ = adfuller(series, regression="c", autolag="AIC")
            kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    except (ValueError, np.linalg.LinAlgError) as exc:
        return {"статус": f"оценка тестов неустойчива: {exc}"}
    return {"ADF": adf_stat, "ADF p": adf_p, "KPSS": kpss_stat, "KPSS p": kpss_p,
            "ADF отвергает единичный корень": adf_p < ALPHA,
            "KPSS отвергает стационарность": kpss_p < ALPHA}


diagnostic_diff = diagnostic.diff().dropna()
display(pd.DataFrame({
    "уровень": stationarity_tests(diagnostic),
    "первая разность": stationarity_tests(diagnostic_diff),
}).T)

In [ ]:
if len(diagnostic_diff) >= 40 and diagnostic_diff.nunique() > 1:
    fig, ax = plt.subplots()
    plot_acf(diagnostic_diff, lags=min(35, len(diagnostic_diff) // 2 - 1), zero=False, ax=ax)
    ax.set(title=f"ACF первой разности ({diagnostic_calendar})", xlabel="Лаг")
    plt.show()
else:
    print("Для ACF нужен более длинный непрерывный ряд с ненулевой дисперсией.")

In [ ]:
if len(diagnostic_diff) >= 40 and diagnostic_diff.nunique() > 1:
    fig, ax = plt.subplots()
    plot_pacf(diagnostic_diff, lags=min(35, len(diagnostic_diff) // 2 - 1),
              zero=False, method="ywm", ax=ax)
    ax.set(title=f"PACF первой разности ({diagnostic_calendar})", xlabel="Лаг")
    plt.show()
else:
    print("Для PACF нужен более длинный непрерывный ряд с ненулевой дисперсией.")

## 5. Календарные признаки и проверка сезонности

Признаки известны на весь горизонт. Добавляем индикаторы дня недели, плавный цикл внутри месяца и флаги начала и конца месяца. Окна 14–16 и 24–26 проверяют возможные эффекты регулярных выплат. При наличии двух лет истории и хотя бы 500 фактов пробуем годовой цикл отдельным кандидатом. Признаки, постоянные на наблюдениях train, исключаем. Зависимость от предыдущих остатков описывает сама ARIMA.

In [ ]:
def calendar_features(dates):
    weekday = pd.Series(dates.day_of_week, index=dates)
    features = pd.get_dummies(weekday, prefix="week", dtype=float).reindex(
        columns=[f"week_{day}" for day in range(1, 7)], fill_value=0.0
    )
    month_phase = 2 * np.pi * (dates.day - 1) / dates.days_in_month
    year_phase = 2 * np.pi * (dates.day_of_year - 1) / (365 + dates.is_leap_year.astype(int))
    return features.assign(
        month_sin=np.sin(month_phase),
        month_cos=np.cos(month_phase),
        month_start=(dates.day <= 3).astype(float),
        month_mid=((dates.day >= 14) & (dates.day <= 16)).astype(float),
        month_25=((dates.day >= 24) & (dates.day <= 26)).astype(float),
        month_end=(dates.days_in_month - dates.day <= 2).astype(float),
        year_sin=np.sin(year_phase),
        year_cos=np.cos(year_phase),
    )


future_dates = pd.date_range(as_of + pd.Timedelta(days=1), as_of + pd.DateOffset(years=1), name="date")
calendar = calendar_features(daily.index.append(future_dates))
observed_train = train.dropna()
varying = calendar.loc[observed_train.index].nunique().loc[lambda count: count > 1].index
short_features = tuple(name for name in varying if not name.startswith("year_"))
annual_features = tuple(name for name in varying if name.startswith("year_"))
has_two_years = (observed_train.index[-1] >= observed_train.index[0] + pd.DateOffset(years=2)
                 and len(observed_train) >= 500)

display(calendar.loc[train.index, list(short_features)].head(7))

In [ ]:
if has_two_years:
    monthly = train.resample("MS").agg(["median", "count", "size"])
    full_month = monthly["size"].eq(monthly.index.days_in_month)
    enough_facts = monthly["count"].ge(.60 * monthly.index.days_in_month)
    monthly_level = monthly["median"].where(full_month & enough_facts)
    monthly_residual = monthly_level - monthly_level.rolling(13, center=True, min_periods=9).median()
    year_profile = monthly_residual.groupby(monthly_residual.index.month).median()

    fig, ax = plt.subplots()
    ax.plot(year_profile.index, year_profile, marker="o", color="#1d6981")
    ax.set(title="Годовой профиль по месячным медианам train",
           xlabel="Месяц", ylabel="Отклонение, млн BYN", xticks=range(1, 13))
    plt.show()
else:
    print("Для годового профиля нужны два года обучающей истории.")

**Тест Уолда с HAC-ковариацией** проверяет, равны ли нулю коэффициенты группы календарных признаков при учёте других групп. Оцениваем регрессию первых разностей уровня на первые разности признаков с константой. Для нескольких групп корректируем p-значения методом Холма. Это диагностическая проверка. Пользу каждого набора для прогноза оцениваем по MAE на valid.

In [ ]:
diagnostic_features = short_features
if has_two_years and len(diagnostic) >= 500:
    diagnostic_features += annual_features

design = calendar.loc[diagnostic.index, list(diagnostic_features)].diff().iloc[1:]
design = design.loc[:, design.nunique().gt(1)]
design = add_constant(design)

rank_ok = len(design) >= max(84, 8 * design.shape[1]) and (
    np.linalg.matrix_rank(design) == design.shape[1]
)
if rank_ok and diagnostic_diff.nunique() > 1 and design.shape[1] > 1:
    calendar_test = OLS(diagnostic_diff, design).fit(cov_type="HAC", cov_kwds={"maxlags": 14})
    effects = []
    for group in ("week", "month", "year"):
        columns = [i for i, column in enumerate(design) if column.startswith(f"{group}_")]
        if columns:
            restriction = np.eye(design.shape[1])[columns]
            result = calendar_test.wald_test(restriction, scalar=True)
            effects.append({"группа": group, "статистика": float(result.statistic), "p": float(result.pvalue)})
    calendar_tests = pd.DataFrame(effects).set_index("группа")
    calendar_tests["p с поправкой Холма"] = multipletests(calendar_tests["p"], method="holm")[1]
    calendar_tests["эффект при 5%"] = calendar_tests["p с поправкой Холма"].lt(ALPHA)
    display(calendar_tests.round(4))
else:
    print("Для календарного теста нужен достаточно длинный непрерывный train и различимые признаки.")

## 6. Обучение на train

Начинаем с небольшого набора порядков. SARIMA использует недельный период 7. Для SARIMAX сравниваем тот же порядок с календарными признаками и, если хватает истории, с добавлением годового цикла. Сходимость проверяем перед расчётом MAE. [Модели ARIMA и SARIMAX в statsmodels](https://www.statsmodels.org/stable/statespace.html)

In [ ]:
from typing import NamedTuple


class ModelSpec(NamedTuple):
    order: tuple
    seasonal_order: tuple = (0, 0, 0, 0)
    columns: tuple = ()


NAIVE = "Последний факт"
WEEKLY = "Недельный наивный"
BASELINES = (NAIVE, WEEKLY)
WEEK_SEASON = (1, 0, 0, 7)
model_specs = {
    "ARIMA(1,0,0)": ModelSpec((1, 0, 0)),
    "ARIMA(0,1,1)": ModelSpec((0, 1, 1)),
    "SARIMA(1,1,0)×(1,0,0,7)": ModelSpec((1, 1, 0), WEEK_SEASON),
}
if short_features:
    model_specs["SARIMAX + календарь"] = ModelSpec((1, 1, 0), WEEK_SEASON, short_features)
if has_two_years and annual_features:
    model_specs["SARIMAX + календарь + год"] = ModelSpec((1, 1, 0), WEEK_SEASON,
                                                        short_features + annual_features)

display(pd.DataFrame.from_dict(model_specs, orient="index", columns=ModelSpec._fields))

In [ ]:
def regressors(name, dates):
    columns = model_specs[name].columns
    return calendar.loc[dates, list(columns)] if columns else None


def fit_model(name, history, start_params=None):
    spec = model_specs[name]
    model = SARIMAX(
        history, exog=regressors(name, history.index),
        order=spec.order, seasonal_order=spec.seasonal_order,
        trend="c" if spec.order[1] == 0 else "n",
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        fit = model.fit(disp=False, maxiter=180, start_params=start_params)
    if not fit.mle_retvals.get("converged") or not np.isfinite(fit.params).all():
        raise ValueError("оптимизация не сошлась")
    return fit

In [ ]:
FIT_ERRORS = (ValueError, RuntimeError, FloatingPointError, np.linalg.LinAlgError)
train_fits = dict.fromkeys(BASELINES)
fit_rows = []

for name in model_specs:
    try:
        train_fits[name] = fit_model(name, train)
        fit_rows.append({"модель": name, "обучение": "сошлась"})
    except FIT_ERRORS as exc:
        fit_rows.append({"модель": name, "обучение": str(exc)})

display(pd.DataFrame(fit_rows).set_index("модель"))

In [ ]:
def forecast_scaled(name, state, history, dates):
    if name == NAIVE:
        return pd.Series(history.dropna().iloc[-1], index=dates)
    if name == WEEKLY:
        recent = history.dropna().iloc[-28:]
        by_weekday = recent.groupby(recent.index.day_of_week).last()
        values = by_weekday.reindex(dates.day_of_week).fillna(recent.iloc[-1]).to_numpy()
        return pd.Series(values, index=dates)
    result = state.get_forecast(steps=len(dates), exog=regressors(name, dates)).predicted_mean
    return pd.Series(result.to_numpy(), index=dates)

In [ ]:
def rolling_errors(name, state, history, holdout, horizon):
    timeline = pd.concat([history, holdout])
    rows = []
    for i in range(len(holdout)):
        observed = timeline.iloc[:len(history) + i]
        origin = observed.index[-1]
        block = holdout.iloc[i:i + horizon]
        predicted = forecast_scaled(name, state, observed, block.index)
        error_byn = ((predicted - block) * SCALE).round(2)
        rows.append(pd.DataFrame({
            "date": block.index, "origin": origin, "lead": (block.index - origin).days,
            "error_byn": error_byn.to_numpy(),
        }))
        if state is not None and i + 1 < len(holdout):
            next_fact = holdout.iloc[i:i + 1]
            state = state.extend(next_fact, exog=regressors(name, next_fact.index))
    return pd.concat(rows, ignore_index=True)

## 7. Выбор по MAE на valid

Коэффициенты обучены на train. В начале каждого дня строим прогноз на следующие 1–30 дней, сохраняя только даты внутри valid. После выпуска фильтр получает факт этого дня. MAE — среднее абсолютное отклонение по всем доступным парам прогноз–факт. Одна целевая дата может участвовать с разной заблаговременностью. [Ретропроверка со скользящей датой выпуска](https://otexts.com/fpp3/toolbox.html#time-series-cross-validation)

Лимит проверяем отдельно. Считаем серии и внутри каждого выпуска, и по последовательным целевым датам на фиксированных горизонтах 1, 7, 14 и 30 дней. Восьмой календарный день подряд с абсолютной ошибкой **свыше** 30 млн BYN означает нарушение. Для пропусков считаем верхнюю границу длины серии, предполагая превышение во все неизвестные дни.

In [ ]:
def longest_streak(mask):
    edges = np.flatnonzero(np.diff(np.r_[False, mask, False]))
    lengths = edges[1::2] - edges[::2]
    if not len(lengths):
        return 0, None, None
    position = int(lengths.argmax())
    return int(lengths[position]), edges[2 * position], edges[2 * position + 1] - 1


def limit_check(error):
    exceeds = error.abs().gt(LIMIT_BYN).to_numpy()
    observed, start, end = longest_streak(exceeds)
    possible, _, _ = longest_streak(exceeds | error.isna().to_numpy())

    status = "passed"
    if observed > MAX_STREAK:
        status = "failed"
    elif len(error) <= MAX_STREAK:
        status = "insufficient_history"
    elif error.count() <= MAX_STREAK or possible > MAX_STREAK:
        status = "insufficient_coverage"

    return {
        "status": status, "max_observed_days": observed, "max_possible_days": possible,
        "start": error.index[start] if start is not None else pd.NaT,
        "end": error.index[end] if end is not None else pd.NaT,
    }

In [ ]:
LIMIT_STATUS_ORDER = ["failed", "insufficient_coverage", "insufficient_history", "passed"]


def worst_status(statuses):
    return min(statuses, key=LIMIT_STATUS_ORDER.index, default="insufficient_history")


def path_limits(panel, horizon):
    sample = panel.loc[panel["lead"].le(horizon)]
    checks = []
    for origin, path in sample.groupby("origin"):
        if len(path) > MAX_STREAK:
            checks.append({"origin": origin, **limit_check(path.set_index("date")["error_byn"])})
    if not checks:
        return {"status": "insufficient_history", "max_observed_days": 0, "origins": 0}

    runs = pd.DataFrame(checks)
    worst = runs.loc[runs["max_observed_days"].idxmax()].to_dict()
    return {**worst, "status": worst_status(runs["status"]),
            "max_possible_days": int(runs["max_possible_days"].max()), "origins": len(runs)}


def lead_report(panel, leads):
    rows = []
    for lead in leads:
        errors = panel.loc[panel["lead"].eq(lead)].set_index("date")["error_byn"].sort_index()
        rows.append({"lead": lead, "mae_byn": errors.abs().mean(),
                     "observations": errors.count(), "calendar_days": len(errors),
                     **limit_check(errors)})
    return pd.DataFrame(rows).set_index("lead")

In [ ]:
validation = {}
rows = []
short_leads = tuple(lead for lead in FIXED_LEADS if lead <= MAE_HORIZON)
for name, fit in train_fits.items():
    panel = rolling_errors(name, fit, train, valid, MAE_HORIZON)
    validation[name] = panel
    path = path_limits(panel, MAE_HORIZON)
    by_lead = lead_report(panel, short_leads)
    statuses = [path["status"], *by_lead["status"]]
    rows.append({"model": name, "mae_byn": panel["error_byn"].abs().mean(),
                 "limit": worst_status(statuses), "observations": panel["error_byn"].count(),
                 "max_observed_days": max(path["max_observed_days"], by_lead["max_observed_days"].max())})

valid_scores = pd.DataFrame(rows).set_index("model").sort_values("mae_byn", kind="stable")
if not np.isfinite(valid_scores["mae_byn"]).any():
    raise ValueError("На valid нет конечных ошибок для сравнения моделей.")
display(valid_scores.round({"mae_byn": 2}))

Из моделей с подтверждённым лимитом выбираем минимальную MAE. Если таких моделей нет, берём минимальную MAE для исследовательского прогноза и явно помечаем допуск как неподтверждённый.

In [ ]:
eligible = valid_scores.loc[valid_scores["limit"].eq("passed")]
approved_on_valid = not eligible.empty
selected_model = (eligible if approved_on_valid else valid_scores).index[0]
print(f"Выбрана модель: {selected_model}")
if not approved_on_valid:
    print("Лимит на valid не подтверждён ни для одной модели. Прогноз исследовательский.")

display(lead_report(validation[selected_model], short_leads).round({"mae_byn": 2}))

In [ ]:
mae_plot = valid_scores["mae_byn"].sort_values(ascending=False) / SCALE
fig, ax = plt.subplots()
colors = ["#c95926" if name == selected_model else "#1d6981" for name in mae_plot.index]
ax.barh(mae_plot.index, mae_plot, color=colors)
ax.set(title="MAE на valid, горизонты 1–30 дней", xlabel="млн BYN")
plt.show()

## 8. Переобучение и диагностика остатков

После выбора переобучаем модель на train + valid. Если оптимизация расходится, сохраняем предыдущие коэффициенты и обновляем состояние фильтра. Порядок и набор признаков остаются прежними.

In [ ]:
def refit_or_filter(name, history, previous, added):
    if previous is None:
        return None
    try:
        return fit_model(name, history, start_params=previous.params)
    except FIT_ERRORS as exc:
        print(f"Переоценка параметров: {exc}. Используем прежние коэффициенты и обновляем состояние.")
        return previous.extend(added, exog=regressors(name, added.index))


train_valid = pd.concat([train, valid])
test_fit = refit_or_filter(selected_model, train_valid, train_fits[selected_model], valid)

**Ljung–Box** проверяет нулевую гипотезу об отсутствии автокорреляции инноваций до указанного лага. Используем непрерывный участок стандартизованных однодневных инноваций до начала test. Первые расчётные дни фильтра исключаем. [Документация Ljung–Box](https://www.statsmodels.org/stable/generated/statsmodels.stats.diagnostic.acorr_ljungbox.html)

In [ ]:
if test_fit is not None:
    spec = model_specs[selected_model]
    innovations = test_fit.filter_results.standardized_forecasts_error[0]
    innovations = pd.Series(innovations, index=test_fit.model.data.row_labels)
    innovations = innovations.where(train_valid.reindex(innovations.index).notna())
    innovations = innovations.iloc[test_fit.loglikelihood_burn:]
    block = longest_complete_block(innovations)
    model_df = spec.order[0] + spec.order[2] + spec.seasonal_order[0] + spec.seasonal_order[2]
    lags = [lag for lag in (7, 14, 28) if model_df < lag <= len(block) // 5]
    if lags and block.nunique() > 1:
        ljung_box = acorr_ljungbox(block, lags=lags, model_df=model_df)
        display(ljung_box.round(4))
    else:
        print("Для Ljung–Box мало непрерывных инноваций с ненулевой дисперсией.")
else:
    print("Выбрана наивная модель. Диагностика инноваций ARIMA здесь пропускается.")

## 9. Финальный test на шести месяцах

Сначала строим прогноз целиком из конца valid. Затем проверяем ежедневный пересчёт с фиксированными коэффициентами и только уже известными фактами. Количество фактов по горизонту указано рядом с MAE. Горизонты за пределами шести месяцев остаются без оценки на этом тесте.

In [ ]:
fixed_scaled = forecast_scaled(selected_model, test_fit, train_valid, test.index)
fixed_byn = (fixed_scaled * SCALE + train_center_byn).rename("forecast_byn")
fixed_error = ((fixed_scaled - test) * SCALE).round(2)
fixed_check = limit_check(fixed_error)

display(pd.DataFrame([{
    "forecast_origin": train_valid.index[-1],
    "mae_byn": fixed_error.abs().mean(),
    "observations": fixed_error.count(),
    **fixed_check,
}]).round({"mae_byn": 2}))

In [ ]:
fig, ax = plt.subplots()
ax.plot(test_byn / SCALE, color="#1d6981", label="Факт")
ax.plot(fixed_byn / SCALE, color="#c95926", label="Прогноз из конца valid")
ax.set(title="Фиксированный прогноз на test", ylabel="млн BYN")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
band = LIMIT_BYN / SCALE
ax.axhspan(-band, band, color="#d7ebdf", alpha=.6)
ax.plot(fixed_error / SCALE, color="#a14029")
ax.axhline(0, color="#555555", linewidth=.8)
ax.set(title="Ошибка фиксированного прогноза на test", ylabel="Прогноз − факт, млн BYN")
plt.show()

In [ ]:
test_panel = rolling_errors(selected_model, test_fit, train_valid, test, horizon=max(FIXED_LEADS))
test_leads = lead_report(test_panel, FIXED_LEADS)
test_limit = path_limits(test_panel, MAE_HORIZON)
test_short = test_panel.loc[test_panel["lead"].le(MAE_HORIZON), "error_byn"]
test_short_status = worst_status([test_limit["status"], *test_leads.loc[list(short_leads), "status"]])

display(pd.DataFrame([{
    **test_limit, "mae_byn": test_short.abs().mean(), "status": test_short_status,
}], index=["Ежедневный пересчёт, 1–30 дней"]).round({"mae_byn": 2}))
display(test_leads.round({"mae_byn": 2}))
if test_short_status != "passed":
    print("Ежедневный пересчёт на test не подтвердил короткий лимит.")

## 10. Годовой прогноз и ГЭПы

Для будущего прогноза модель обучается на всей доступной истории. Центр и масштаб остаются прежними. Изменение остатка используется как приближение чистого потока. При наличии капитализации или иных безденежных изменений понадобится отдельная корректировка потока.

In [ ]:
final_fit = refit_or_filter(selected_model, target, test_fit, test)
future_scaled = forecast_scaled(selected_model, final_fit, target, future_dates)
future_byn = (future_scaled * SCALE + train_center_byn).round(2)
opening_byn = round(daily.iloc[-1], 2)

forecast_df = future_byn.rename("forecast_byn").to_frame()
forecast_df["lead"] = (future_dates - as_of).days
forecast_df["net_flow_byn"] = np.diff(np.r_[opening_byn, future_byn]).round(2)
forecast_df["cumulative_gap_byn"] = (future_byn - opening_byn).round(2)
forecast_df["funding_need_byn"] = (-forecast_df["cumulative_gap_byn"]).clip(lower=0)

if not approved_on_valid or test_short_status != "passed" or fixed_check["status"] != "passed":
    print("Годовой прогноз исследовательский: лимит на контрольных отрезках не подтверждён полностью.")
display(forecast_df.head(7))

In [ ]:
fig, ax = plt.subplots()
ax.plot(daily.iloc[-90:] / SCALE, color="#1d6981", label="Последние факты")
ax.plot(forecast_df["forecast_byn"] / SCALE, color="#c95926", label="Годовой прогноз")
ax.axvline(as_of, color="#555555", linewidth=.8)
ax.set(title="Прогноз остатка на год", ylabel="млн BYN")
ax.legend()
plt.show()

Пиковая потребность — наибольшее падение остатка от последнего факта внутри интервала. Оценка относится к средствам физлиц в BYN. Для ГЭПа банка нужны также потоки по остальным существенным позициям. [Принципы Базельского комитета](https://www.bis.org/committees/bcbs/basel-consolidated-guidelines/module/lqy/10)

In [ ]:
bucket_labels = ["1–7", "8–14", "15–30", "31–90", "91–180", "181–год"]
interval = pd.cut(forecast_df["lead"], [0, *BUCKET_ENDS], labels=bucket_labels)
gaps_df = forecast_df.assign(date=forecast_df.index).groupby(interval, observed=True).agg(
    first_date=("date", "first"),
    last_date=("date", "last"),
    closing_balance_byn=("forecast_byn", "last"),
    net_flow_byn=("net_flow_byn", "sum"),
    cumulative_gap_byn=("cumulative_gap_byn", "last"),
    peak_funding_byn=("funding_need_byn", "max"),
)
display(gaps_df.rename_axis("горизонт").round({"net_flow_byn": 2}))

In [ ]:
fig, ax = plt.subplots()
ax.plot(forecast_df["funding_need_byn"] / SCALE, color="#a14029")
ax.set(title="Дневная потребность в замещении средств", ylabel="млн BYN")
plt.show()